# Atlas Python Static Analysis Tutorial

Atlas builds a complete navigable tree of your Python codebase through AST-based reconnaissance.

## 1. Basic Setup

Build a project tree by pointing Atlas at your target directory:

In [1]:
from analyzer import build_complete_atlas

# Build complete project tree
project = build_complete_atlas('sample_files')
print(f"Project: {project.name}")

Project: sample_files


## 2. Visualize Project Structure

Display the complete project tree:

In [2]:
# Print full tree structure
project.print()

Project(sample_files)
  Package(api)
    Package(endpoints)
      Module(product_endpoints)
        Class(ProductEndpoints)
          Function(__init__)
            Argument(self)
              MissingArgumentTypeHint(ArgumentNode)
          Function(create_category)
            Argument(self)
              MissingArgumentTypeHint(ArgumentNode)
            Argument(name)
            Argument(description)
              MissingArgumentTypeHint(ArgumentNode)
          Function(get_category)
            Argument(self)
              MissingArgumentTypeHint(ArgumentNode)
            Argument(category_id)
              MissingArgumentTypeHint(ArgumentNode)
          Function(create_product)
            Argument(self)
              MissingArgumentTypeHint(ArgumentNode)
            Argument(name)
            Argument(price)
            Argument(category_id)
            Argument(description)
              MissingArgumentTypeHint(ArgumentNode)
          Function(get_product)
            Argument(

## 3. Navigation API

Navigate through packages, modules, classes, and functions:

In [3]:
# List all packages
packages = project.list_packages()
print(f"Found {len(packages)} packages")

# Get specific package
models_pkg = project.get_package('models')
print(f"Package: {models_pkg.fqn}")

# List modules in package
modules = models_pkg.list_modules()
print(f"Modules: {[m.name for m in modules]}")

Found 5 packages
Package: sample_files.models
Modules: ['order', 'product', 'user']


## 4. Class Discovery

Explore classes and their methods:

In [4]:
# Get a module and list its classes
user_module = models_pkg.get_module('user')
classes = user_module.list_classes()
print(f"Classes in {user_module.name}: {[c.name for c in classes]}")

# Get specific class
user_class = user_module.get_class('User')
print(f"\nClass: {user_class.fqn}")

# List methods
methods = user_class.list_methods()
print(f"Methods: {[m.name for m in methods]}")

Classes in user: ['User', 'UserProfile']

Class: sample_files.models.user.User
Methods: ['__init__', 'get_email', 'set_email', 'add_role', 'has_role', 'get_roles', 'activate', 'deactivate']


## 5. Function Signatures

Analyze function arguments and return types:

In [5]:
# Get a method and examine its signature
init_method = user_class.get_method('__init__')
print(f"Method: {init_method.name}")

# List arguments
args = init_method.list_arguments()
print(f"\nArguments ({len(args)}):")
for arg in args:
    type_info = arg._type if hasattr(arg, '_type') and arg._type else None
    type_str = type_info.name if type_info else "No type hint"
    print(f"  {arg.name}: {type_str}")

# Check return type
returns = init_method.list_returns()
if returns:
    ret = returns[0]
    type_info = ret._type if hasattr(ret, '_type') and ret._type else None
    print(f"\nReturn type: {type_info.name if type_info else 'None'}")

Method: __init__

Arguments (5):
  self: No type hint
  user_id: str
  email: str
  username: No type hint
  password: No type hint


## 6. Attribute Discovery

Find class and instance attributes:

In [6]:
# List class attributes
class_attrs = user_class.list_class_attributes()
print(f"Class attributes ({len(class_attrs)}):")
for attr in class_attrs:
    print(f"  {attr.name}")

# List instance attributes
instance_attrs = user_class.list_instance_attributes()
print(f"\nInstance attributes ({len(instance_attrs)}):")
for attr in instance_attrs:
    type_info = attr._type if hasattr(attr, '_type') and attr._type else None
    type_str = type_info.name if type_info else "No type hint"
    print(f"  {attr.name}: {type_str}")

Class attributes (0):

Instance attributes (5):
  email: No type hint
  username: No type hint
  password: No type hint
  is_active: No type hint
  roles: No type hint


## 7. Import Analysis

Discover module imports:

In [7]:
# List all imports in a module
imports = user_module.list_imports()
print(f"Found {len(imports)} import statements")

# Examine individual imports
for imp in imports[:3]:  # Show first 3
    aliases = imp.list_aliases()
    for alias in aliases:
        # Access AST data for import details
        original = alias.source_data.name
        local = alias.source_data.asname if alias.source_data.asname else original
        if local != original:
            print(f"  {local} (alias for {original})")
        else:
            print(f"  {original}")

Found 3 import statements
  Optional
  List
  datetime
  BaseEntity


## 8. Violation Detection

Find type hint violations and code issues:

In [8]:
# Check for violations on arguments
violations = []
for arg in args:
    if hasattr(arg, '_violations') and arg._violations:
        violations.extend(arg._violations)

print(f"Found {len(violations)} violations in {init_method.name}")
for v in violations[:3]:  # Show first 3
    print(f"  {v.__class__.__name__}")

Found 3 violations in __init__
  MissingArgumentTypeHint
  MissingArgumentTypeHint
  MissingArgumentTypeHint


## 9. Recursive Discovery

Use `list_all_*()` methods to traverse the entire tree:

In [9]:
# Find all classes in the entire project
all_classes = project.list_all_classes()
print(f"Total classes in project: {len(all_classes)}")

# Find all methods across all classes
all_methods = project.list_all_methods()
print(f"Total methods in project: {len(all_methods)}")

# Find all functions (module-level)
all_functions = project.list_all_functions()
print(f"Total functions in project: {len(all_functions)}")

Total classes in project: 27
Total methods in project: 142
Total functions in project: 15


## 10. JSON Serialization

Export project analysis to JSON:

In [10]:
# Export to JSON file
project.save_dump('project_analysis.json')

# Or get as dict for programmatic use
data = project.dump()
print(f"Project type: {data['type']}")
print(f"Project name: {data['name']}")
print(f"Number of packages: {len(data.get('children', []))}")

✓ Project tree serialized to: project_analysis.json
  File size: 79,756 bytes
  Nodes serialized: 259
Project type: Project
Project name: sample_files
Number of packages: 6


## 11. Practical: Find All Type Violations

Scan entire project for missing type hints:

In [11]:
# Collect all violations across the project
all_violations = []

# Check all arguments
for arg in project.list_all_arguments():
    if hasattr(arg, '_violations') and arg._violations:
        for v in arg._violations:
            all_violations.append({
                'type': v.__class__.__name__,
                'location': arg.fqn,
                'entity': 'argument'
            })

# Check all returns
for ret in project.list_all_returns():
    if hasattr(ret, '_violations') and ret._violations:
        for v in ret._violations:
            all_violations.append({
                'type': v.__class__.__name__,
                'location': ret.fqn,
                'entity': 'return'
            })

print(f"Total violations: {len(all_violations)}")
print(f"\nBreakdown:")
from collections import Counter
violation_counts = Counter(v['type'] for v in all_violations)
for vtype, count in violation_counts.most_common():
    print(f"  {vtype}: {count}")

Total violations: 236

Breakdown:
  MissingArgumentTypeHint: 236


## 12. Practical: Analyze Method Complexity

Find methods with many arguments (potential complexity issues):

In [12]:
# Analyze all methods for argument count
complex_methods = []

for method in project.list_all_methods():
    args = method.list_arguments()
    # Filter out 'self' and 'cls'
    param_count = len([a for a in args if a.name not in ('self', 'cls')])
    
    if param_count >= 4:  # Threshold for "complex"
        complex_methods.append({
            'fqn': method.fqn,
            'params': param_count
        })

# Sort by parameter count
complex_methods.sort(key=lambda x: x['params'], reverse=True)

print(f"Found {len(complex_methods)} methods with 4+ parameters\n")
for m in complex_methods[:5]:  # Show top 5
    print(f"  {m['fqn']}: {m['params']} params")

Found 10 methods with 4+ parameters

  sample_files.models.product.Product.__init__: 5 params
  sample_files.services.email_service.EmailService.send_email: 5 params
  sample_files.services.payment_service.PaymentProcessor.process_payment: 5 params
  sample_files.api.endpoints.product_endpoints.ProductEndpoints.create_product: 4 params
  sample_files.models.user.User.__init__: 4 params


In [13]:
"""
Test Analysis Phase Step 2: Verify analyze() cascade works

This test verifies that:
1. project.analyze() can be called without errors
2. No NotImplementedError is raised from any node
"""

from analyzer import build_complete_atlas

def test_analyze_cascade():
    """Test that analyze() cascades through entire tree without errors."""
    
    print("=" * 70)
    print("Analysis Phase Step 2 Test: analyze() Cascade")
    print("=" * 70)
    
    # Build project tree
    print("\n1. Building project tree...")
    project = build_complete_atlas("sample_files")
    print(f"   ✓ Project '{project.name}' built successfully")
    
    # Show tree statistics using navigation API
    print(f"\n2. Tree statistics (direct children):")
    print(f"   Packages: {len(project.list_packages())}")
    print(f"   Modules: {len(project.list_modules())}")
    print(f"   All classes (recursive): {len(project.list_all_classes())}")
    print(f"   All functions (recursive): {len(project.list_all_functions())}")
    
    # Call analyze() - this is the critical test
    print(f"\n3. Calling project.analyze()...")
    try:
        project.analyze()
        print("   ✓ analyze() completed without errors!")
    except NotImplementedError as e:
        print(f"   ✗ FAILED: {e}")
        return False
    except Exception as e:
        print(f"   ✗ UNEXPECTED ERROR: {e}")
        import traceback
        traceback.print_exc()
        return False
    
    # Verify tree is still intact
    print(f"\n4. Verifying tree integrity...")
    classes_after = len(project.list_all_classes())
    functions_after = len(project.list_all_functions())
    print(f"   ✓ Tree intact: {classes_after} classes, {functions_after} functions")
    
    # Success!
    print("\n" + "=" * 70)
    print("✓ STEP 2 TEST PASSED!")
    print("=" * 70)
    print("\nAll nodes successfully implement analyze() method.")
    print("Ready to proceed to Step 3: Create BaseNote infrastructure")
    print()
    
    return True

if __name__ == "__main__":
    success = test_analyze_cascade()
    exit(0 if success else 1)

Analysis Phase Step 2 Test: analyze() Cascade

1. Building project tree...
   ✓ Project 'sample_files' built successfully

2. Tree statistics (direct children):
   Packages: 5
   Modules: 1
   All classes (recursive): 27
   All functions (recursive): 15

3. Calling project.analyze()...
   Found annotated assignment: VERSION: ... = ... (line 31)
   Found annotated assignment: MAX_CONNECTIONS: ... = ... (line 32)
   Found annotated assignment: DEBUG_ENABLED: ... = ... (line 33)
   Found annotated assignment: TIMEOUT_SECONDS: ... = ... (line 34)
   Found assignment: author = ... (line 40)
   Found assignment: default_port = ... (line 41)
   Found assignment: is_production = ... (line 42)
   Found assignment: retry_delay = ... (line 43)
   Found assignment: error_codes = ... (line 49)
   Found assignment: default_headers = ... (line 50)
   Found assignment: allowed_methods = ... (line 51)
   Found assignment: coordinate = ... (line 52)
   Found assignment: token_manager = ... (line 59)
   

In [14]:
"""Test Analysis Phase Step 3: BaseNote infrastructure."""

from analyzer.builder import build_complete_atlas
from analyzer.analysis import BaseNote


# Create a simple concrete note class for testing
class TestNote(BaseNote):
    """Simple test note with a message."""
    def __init__(self, node, message):
        super().__init__(node)
        self.message = message


def main():
    print("Testing Analysis Phase Step 3: BaseNote Infrastructure\n")
    print("=" * 60)
    
    # Build project tree
    print("\n1. Building project tree...")
    project = build_complete_atlas("sample_files/")
    print(f"   ✓ Project built: {project.name}")
    
    # Test 1: Verify _notes collection exists on all nodes
    print("\n2. Verifying _notes collection on nodes...")
    assert hasattr(project, '_notes'), "Project should have _notes"
    assert isinstance(project._notes, list), "_notes should be a list"
    print(f"   ✓ Project has _notes: {project._notes}")
    
    # Get a node to test with (find any module with a function, or use project)
    test_node = None
    for module in project._modules:
        if module._functions:
            test_node = module._functions[0]
            break
    
    if test_node is None:
        if project._modules:
            test_node = project._modules[0]
            print("   ⚠ No functions found in sample_files, testing with module instead")
        else:
            test_node = project
            print("   ⚠ No modules found, testing with project node")
        
    assert hasattr(test_node, '_notes'), "Node should have _notes"
    print(f"   ✓ {test_node.name} has _notes: {test_node._notes}")
    
    # Test 2: Create and attach notes
    print("\n3. Creating and attaching test notes...")
    note1 = TestNote(test_node, "First note")
    note2 = TestNote(test_node, "Second note")
    test_node._notes.append(note1)
    test_node._notes.append(note2)
    print(f"   ✓ Attached 2 notes to {test_node.name}")
    
    # Test 3: Query all notes
    print("\n4. Querying notes...")
    all_notes = test_node.get_notes()
    assert len(all_notes) == 2, f"Expected 2 notes, got {len(all_notes)}"
    print(f"   ✓ get_notes() returned {len(all_notes)} notes")
    for note in all_notes:
        print(f"      - {note}")
    
    # Test 4: Query filtered notes
    print("\n5. Querying filtered notes...")
    test_notes = test_node.get_notes(TestNote)
    assert len(test_notes) == 2, f"Expected 2 TestNotes, got {len(test_notes)}"
    print(f"   ✓ get_notes(TestNote) returned {len(test_notes)} notes")
    
    # Test 5: Create another note type for filtering
    class DifferentNote(BaseNote):
        pass
    
    diff_note = DifferentNote(test_node)
    test_node._notes.append(diff_note)
    
    print("\n6. Testing note type filtering...")
    all_notes = test_node.get_notes()
    test_notes = test_node.get_notes(TestNote)
    diff_notes = test_node.get_notes(DifferentNote)
    
    assert len(all_notes) == 3, f"Expected 3 total notes, got {len(all_notes)}"
    assert len(test_notes) == 2, f"Expected 2 TestNotes, got {len(test_notes)}"
    assert len(diff_notes) == 1, f"Expected 1 DifferentNote, got {len(diff_notes)}"
    
    print(f"   ✓ Total notes: {len(all_notes)}")
    print(f"   ✓ TestNote filtered: {len(test_notes)}")
    print(f"   ✓ DifferentNote filtered: {len(diff_notes)}")
    
    # Test 6: Verify notes across tree
    print("\n7. Verifying _notes exist across entire tree...")
    node_count = 0
    nodes_with_notes = 0
    
    def count_nodes(node):
        nonlocal node_count, nodes_with_notes
        node_count += 1
        assert hasattr(node, '_notes'), f"{node} missing _notes"
        nodes_with_notes += 1
        
        # Recursively check all children
        for child in node._get_direct_children():
            count_nodes(child)
    
    count_nodes(project)
    print(f"   ✓ Verified {node_count} nodes")
    print(f"   ✓ All {nodes_with_notes} nodes have _notes collection")
    
    print("\n" + "=" * 60)
    print("✓ Step 3 Complete: BaseNote Infrastructure Working!")
    print("\nNext: Step 4 will add analysis visitors")


if __name__ == '__main__':
    main()

Testing Analysis Phase Step 3: BaseNote Infrastructure


1. Building project tree...
   ✓ Project built: sample_files

2. Verifying _notes collection on nodes...
   ✓ Project has _notes: []
   ⚠ No functions found in sample_files, testing with module instead
   ✓ atlas_testbed has _notes: []

3. Creating and attaching test notes...
   ✓ Attached 2 notes to atlas_testbed

4. Querying notes...
   ✓ get_notes() returned 2 notes
      - TestNote(node=atlas_testbed)
      - TestNote(node=atlas_testbed)

5. Querying filtered notes...
   ✓ get_notes(TestNote) returned 2 notes

6. Testing note type filtering...
   ✓ Total notes: 3
   ✓ TestNote filtered: 2
   ✓ DifferentNote filtered: 1

7. Verifying _notes exist across entire tree...
   ✓ Verified 1272 nodes
   ✓ All 1272 nodes have _notes collection

✓ Step 3 Complete: BaseNote Infrastructure Working!

Next: Step 4 will add analysis visitors


In [15]:
"""Test ModuleAnalysisVisitor - Minimal skeleton test."""

import tempfile
import os
from pathlib import Path
from analyzer.builder import build_complete_atlas


def create_test_module():
    """Create a temporary test module with simple assignments."""
    test_code = '''
# Simple test module

# Simple assignments
x = 5
name = "hello"
is_active = True

# Annotated assignment
count: int = 100
'''
    
    # Create temporary directory and file
    temp_dir = tempfile.mkdtemp()
    test_file = Path(temp_dir) / "test_module.py"
    test_file.write_text(test_code)
    
    return temp_dir, test_file


def main():
    print("Testing ModuleAnalysisVisitor (Minimal Skeleton)\n")
    print("=" * 60)
    
    # Create test module
    print("\n1. Creating test module...")
    temp_dir, test_file = create_test_module()
    print(f"   ✓ Created: {test_file}")
    
    try:
        # Build project
        print("\n2. Building project tree...")
        project = build_complete_atlas(temp_dir)
        print(f"   ✓ Project built: {project.name}")
        
        # Get the test module
        print("\n3. Finding test module...")
        test_module = None
        for module in project._modules:
            if module.name == "test_module":
                test_module = module
                break
        
        if test_module is None:
            print("   ✗ Could not find test_module")
            return
        
        print(f"   ✓ Found module: {test_module.name}")
        
        # Run analysis
        print("\n4. Running analysis on module...")
        print("   Visitor output:")
        test_module.analyze()
        print("   ✓ Analysis complete")
        
        # Verify visitor ran by checking it found assignments
        from analyzer.analysis.visitors import ModuleAnalysisVisitor
        
        print("\n5. Verifying visitor worked...")
        visitor = ModuleAnalysisVisitor(test_module)
        visitor.visit(test_module.source_data.ast_node)
        
        print(f"   ✓ Visitor found {visitor.assignment_count} assignments")
        
        if visitor.assignment_count >= 4:  # x, name, is_active, count
            print("   ✓ Found expected number of assignments")
        else:
            print(f"   ⚠ Expected at least 4 assignments, found {visitor.assignment_count}")
        
        print("\n" + "=" * 60)
        print("✓ ModuleAnalysisVisitor Skeleton Test Complete!")
        print("\nNext steps:")
        print("  - Add type inference logic")
        print("  - Create TypeNote class")
        print("  - Attach notes to module node")
        
    finally:
        # Cleanup
        import shutil
        shutil.rmtree(temp_dir)
        print(f"\n✓ Cleaned up temporary directory")


if __name__ == '__main__':
    main()

Testing ModuleAnalysisVisitor (Minimal Skeleton)


1. Creating test module...
   ✓ Created: C:\Users\wilha\AppData\Local\Temp\tmpw97zn84j\test_module.py

2. Building project tree...
   ✓ Project built: tmpw97zn84j

3. Finding test module...
   ✓ Found module: test_module

4. Running analysis on module...
   Visitor output:
   Found assignment: x = ... (line 5)
   Found assignment: name = ... (line 6)
   Found assignment: is_active = ... (line 7)
   Found annotated assignment: count: ... = ... (line 10)
   ✓ Analysis complete

5. Verifying visitor worked...
   Found assignment: x = ... (line 5)
   Found assignment: name = ... (line 6)
   Found assignment: is_active = ... (line 7)
   Found annotated assignment: count: ... = ... (line 10)
   ✓ Visitor found 4 assignments
   ✓ Found expected number of assignments

✓ ModuleAnalysisVisitor Skeleton Test Complete!

Next steps:
  - Add type inference logic
  - Create TypeNote class
  - Attach notes to module node

✓ Cleaned up temporary direct

In [16]:
"""
Test code for TypeInferenceEngine.linearize() method
Copy and paste this into a Jupyter notebook cell
"""

import ast
from analyzer.analysis.expression_traversal import TypeInferenceEngine

# Create a mock project node (linearize doesn't actually use it yet)
class MockProjectNode:
    pass

project = MockProjectNode()
engine = TypeInferenceEngine(project)

# Test cases with increasing complexity
test_expressions = [
    # Simple name
    "user",
    
    # Simple attribute access
    "user.name",
    
    # Chained attribute access
    "user.profile.email",
    
    # Simple method call
    "user.validate()",
    
    # Method call with attribute
    "user.profile.get_status()",
    
    # Complex chain
    "admin_user.email.lower().strip()",
    
    # Very complex chain
    "payment_service.get_processor().get_provider_name()",
    
    # List indexing
    "error_codes[0]",
    
    # Dictionary access
    "config_dict['host']",
    
    # Attribute then subscript
    "user.permissions[0]",
    
    # Subscript then attribute
    "users[0].name",
    
    # Method call then subscript
    "get_users()[0]",
    
    # Complex: subscript, method, subscript
    "data[0].get_items()[5]",
    
    # Multi-dimensional indexing
    "matrix[i][j]",
    
    # Dictionary in method chain
    "config.get_settings()['database']['host']",
]

print("=" * 70)
print("TypeInferenceEngine.linearize() Test Results")
print("=" * 70)

for expr_string in test_expressions:
    print(f"\nExpression: {expr_string}")
    print("-" * 70)
    
    # Parse the expression
    expr_ast = ast.parse(expr_string, mode='eval').body
    
    # Linearize it
    loq = engine.linearize(expr_ast)
    
    # Display the Linear Operation Queue
    print(f"LOQ ({len(loq)} operations):")
    for i, op in enumerate(loq, 1):
        print(f"  {i}. {op}")
    
print("\n" + "=" * 70)
print("Test Complete!")
print("=" * 70)

TypeInferenceEngine.linearize() Test Results

Expression: user
----------------------------------------------------------------------
LOQ (1 operations):
  1. GetName('user')

Expression: user.name
----------------------------------------------------------------------
LOQ (2 operations):
  1. GetName('user')
  2. GetAttribute('name')

Expression: user.profile.email
----------------------------------------------------------------------
LOQ (3 operations):
  1. GetName('user')
  2. GetAttribute('profile')
  3. GetAttribute('email')

Expression: user.validate()
----------------------------------------------------------------------
LOQ (3 operations):
  1. GetName('user')
  2. GetAttribute('validate')
  3. CallFunction()

Expression: user.profile.get_status()
----------------------------------------------------------------------
LOQ (4 operations):
  1. GetName('user')
  2. GetAttribute('profile')
  3. GetAttribute('get_status')
  4. CallFunction()

Expression: admin_user.email.lower().str